# Pitch Uncertainty Visualization (The Cloud)

This notebook visualizes the uncertainty of a pitch trajectory. Instead of a single line, we generate a 'cloud' of possible paths based on small variations in the initial conditions (release speed, release point, movement).

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Add parent directory to path so we can import src
sys.path.append(os.path.abspath('../'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from src.data_handler import StatcastDataHandler
from src.physics import calculate_trajectory
from src.uncertainty import generate_pitch_cloud
from src.visualization import visualize_pitch_cloud

## Load Data

In [2]:
# Load a sample of data
handler = StatcastDataHandler()
df = handler.fetch_statcast(start_year=2024, end_year=2024)

if df.empty:
    print("2024 empty, trying 2025...")
    df = handler.fetch_statcast(start_year=2025, end_year=2025)
    
print(f"Loaded {len(df)} rows")

Loading data from 2024 to 2024...
Loaded 722296 rows


## Select a Pitch

In [12]:
# --- FORCE REFRESH FOR CORBIN BURNES ---
target_name = 'Snell, Blake'

# 1. Selection
try:
    top_pitcher = df[df['player_name'] == target_name]['pitcher'].iloc[0]
except IndexError:
    print(f"ERROR: {target_name} not found! Falling back to mode.")
    top_pitcher = df['pitcher'].mode()[0]

pitcher_name = df[df['pitcher'] == top_pitcher]['player_name'].iloc[0]
print(f"CONFIRMED SELECTED PITCHER: {pitcher_name}")

# 2. Get Data
pitcher_df = df[df['pitcher'] == top_pitcher]
pitch_type = pitcher_df['pitch_type'].mode()[0]
row = pitcher_df[pitcher_df['pitch_type'] == pitch_type].iloc[0] # Take first pitch

# 3. Generate & Visualize
cloud_trajs = generate_pitch_cloud(row, n_samples=200)
actual_traj = calculate_trajectory(row)

fig = visualize_pitch_cloud(actual_traj, cloud_trajs, pitch_type=pitch_type)
fig.show()

CONFIRMED SELECTED PITCHER: Snell, Blake


In [10]:
# --- CORRECTED SELECTION LOGIC ---
pitcher_name = 'Snell, Blake'  # CHANGE THIS NAME TO SWITCH PITCHERS

# Find the ID for this name
try:
    top_pitcher = df[df['player_name'] == pitcher_name]['pitcher'].iloc[0]
except IndexError:
    print(f"Visualize Error: {pitcher_name} not found in this dataset!")
    # Fallback
    top_pitcher = df['pitcher'].mode()[0]
    pitcher_name = df[df['pitcher'] == top_pitcher]['player_name'].iloc[0]
    print(f"Falling back to: {pitcher_name}")

print(f"Selected Pitcher: {pitcher_name} (ID: {top_pitcher})")

# Filter for data
pitcher_df = df[df['pitcher'] == top_pitcher]

# Pick their top pitch type
pitch_type = pitcher_df['pitch_type'].mode()[0]
print(f"Analyzing {pitch_type} (n={len(pitcher_df[pitcher_df['pitch_type'] == pitch_type])})")

# Select the specific pitch to visualize (median extension)
pitch_subset = pitcher_df[pitcher_df['pitch_type'] == pitch_type]
row = pitch_subset.iloc[len(pitch_subset)//2]

Selected Pitcher: Snell, Blake (ID: 605483)
Analyzing FF (n=822)


## Generate Cloud
We generate 50 variations of this pitch.

In [4]:
# --- ONE-STEP GENERATION & VISUALIZATION (INVERTED X) ---
n_samples = 200
print(f"Generating Cloud for: {pitcher_name} ({pitch_type})...")

# 1. GENERATE (Forces refresh)
cloud_trajs = generate_pitch_cloud(row, n_samples=n_samples)
actual_traj = calculate_trajectory(row)

# 2. VISUALIZE (With User-Preferred Inverted X)
fig = visualize_pitch_cloud(actual_traj, cloud_trajs, pitch_type=pitch_type)
fig.show()

Generating Cloud for: Snell, Blake (FF)...


## Visualize 3D Cloud

In [5]:
if not df.empty:
    fig = visualize_pitch_cloud(actual_traj, cloud_trajs, pitch_type=pitch_type)

    # 1. Plot Cloud (Thin, transparent lines)
    for i, traj in enumerate(cloud_trajs):
        fig.add_trace(go.Scatter3d(
            x=traj['x'], y=traj['y'], z=traj['z'],
            mode='lines',
            line=dict(color='rgba(100, 100, 255, 0.15)', width=2),
            name='Cloud' if i == 0 else None,
            showlegend=(i == 0)
        ))

    # 2. Plot Actual Trajectory (Thick, solid)
    if actual_traj is not None:
        fig.add_trace(go.Scatter3d(
            x=actual_traj['x'], y=actual_traj['y'], z=actual_traj['z'],
            mode='lines',
            line=dict(color='red', width=6),
            name='Actual Pitch'
        ))

    # 3. Add Plate and Zone
    # Plate (Home Plate is approx 17 inches wide, pentagon shape... simplified here)
    y_front = 17.0 / 12.0  # 1.4167 ft (front of plate in Statcast coordinates)
    plate_x = [-0.708, 0.708, 0.708, 0, -0.708, -0.708]
    plate_y = [y_front, y_front, y_front - 0.708, 0, y_front - 0.708, y_front]
    fig.add_trace(go.Scatter3d(
        x=plate_x, y=plate_y, z=[0]*6,
        mode='lines',
        line=dict(color='black', width=4),
        name='Home Plate'
    ))

    # Strike Zone (Rectangular box at front of plate y=y_front)
    sz_top = row.get('sz_top', 3.5)
    sz_bot = row.get('sz_bot', 1.5)
    zone_x = [-0.708, 0.708, 0.708, -0.708, -0.708]
    zone_z = [sz_bot, sz_bot, sz_top, sz_top, sz_bot]
    fig.add_trace(go.Scatter3d(
        x=zone_x, y=[y_front]*5, z=zone_z,
        mode='lines',
        line=dict(color='black', width=4),
        name='Strike Zone'
    ))

    # Layout
    fig.update_layout(
        title=f"Uncertainty Cloud: {pitcher_name} {pitch_type}",
        scene=dict(
            xaxis=dict(title='X (ft)', range=[-3, 3]),
            yaxis=dict(title='Distance (ft)', range=[0, 60.5]),
            zaxis=dict(title='Height (ft)', range=[0, 6]),
            aspectmode='manual',
            aspectratio=dict(x=1, y=10, z=1)
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()

## 2D Distribution at Plate
Let's see where the pitches cross the front of the plate (y=0).

In [6]:
if not df.empty:
    final_pts = []
    for traj in cloud_trajs:
        # Get the point closest to y=0 (or exactly y=0 if we interpolated)
        # Our calc_traj usually stops at t_final which is at y_plate (y=1.417 or 0 depending on ref)
        # Let's assume the last point is the plate crossing
        final_pts.append(traj.iloc[-1])

    df_cloud = pd.DataFrame(final_pts)

    fig2 = go.Figure()

    # Scatter of cloud points
    fig2.add_trace(go.Scatter(
        x=df_cloud['x'], y=df_cloud['z'],
        mode='markers',
        marker=dict(color='blue', size=8, opacity=0.5),
        name='Possible Locations'
    ))

    # Actual Location
    if actual_traj is not None:
        last = actual_traj.iloc[-1]
        fig2.add_trace(go.Scatter(
            x=[last['x']], y=[last['z']],
            mode='markers',
            marker=dict(color='red', size=12, symbol='x'),
            name='Actual'
        ))

    # Strike Zone Box
    fig2.add_shape(type="rect",
        x0=-0.708, y0=sz_bot, x1=0.708, y1=sz_top,
        line=dict(color="Black"),
        fillcolor="rgba(0,0,0,0)",
    )

    fig2.update_layout(
        title=f"Pitch Location Uncertainty (At Plate) - {pitcher_name}",
        xaxis=dict(title='Horizontal Location (ft)', range=[-2, 2]),
        yaxis=dict(title='Vertical Location (ft)', range=[0, 5], scaleanchor="x", scaleratio=1),
    )

    fig2.show()